# 가설검증: A/B/C군 연속형 변수 vs TARGET(연체 여부) — Welch's t-test

절차:
1. 연속형 변수만 대상 (범주형·이진 변수는 카이제곱으로 별도 진행)
2. H0: TARGET=0/1 두 집단의 평균이 같다 vs H1: 다르다 (양측검정)
3. Levene's test로 등분산 여부 확인(참고용 — 결과와 무관하게 Welch's 사용)
4. 왜도(skewness) 확인 — 표본이 커서(30만+) CLT로 강건하지만 극단적 왜도는 별도 표시
5. Welch's t-test(equal_var=False) 실행
6. 여러 변수를 한꺼번에 검정하므로 FDR(Benjamini-Hochberg) 보정
7. p-value만이 아니라 효과크기(Cohen's d)도 함께 산출
8. 변수선정_최종.csv의 IV값과 병합해 최종표 구성

## 0. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 160)

BASE_DIR = "/content/drive/MyDrive/BOOSTMAP/데이터/완료"  # 본인 환경에 맞게 수정

FILES = {
    "A_금융":       os.path.join(BASE_DIR, "model_A_financial_final.csv"),
    "B_기존비금융": os.path.join(BASE_DIR, "model_B_nonfinancial_final.csv"),
    "C_신규비금융": os.path.join(BASE_DIR, "model_C_nonfinancial_new_final.csv"),
}
# IV값이 들어있는 변수선정 파일 — 경로가 다르면 아래 한 줄만 수정
VARSEL_PATH = os.path.join(BASE_DIR, "변수선정_최종.csv")

TARGET_COL = "TARGET"
ID_COL = "SK_ID_CURR"
LOW_CARD_THRESHOLD = 15    # 이 값 이하 고유값이면 연속형이 아니라 범주형/이진으로 간주
SKEW_ALERT = 2.0           # 이 값을 넘으면 "왜도 큼"으로 표시(해석 시 유의)
ALPHA = 0.05

## 1. 연속형 컬럼 판별 함수

In [ ]:
def get_continuous_cols(df, id_col=ID_COL, target_col=TARGET_COL, threshold=LOW_CARD_THRESHOLD):
    """수치형이면서 고유값이 threshold개를 초과하는 컬럼만 연속형으로 분류.
    범주형/이진(원-핫 인코딩 포함)은 여기서 제외 — 카이제곱 검정을 따로 써야 함."""
    cols = [c for c in df.columns if c not in (id_col, target_col)]
    continuous = []
    for c in cols:
        s = df[c]
        if s.dtype == object:
            continue
        if s.nunique(dropna=True) > threshold:
            continuous.append(c)
    return continuous

## 2. 변수 하나에 대한 전체 검정 파이프라인

In [ ]:
def test_one_variable(x1, x0):
    """x1 = TARGET==1 그룹, x0 = TARGET==0 그룹.
    반환: Levene's p, 왜도(전체 표본), Welch's t/p, Cohen's d"""

    # (a) 등분산 검정 — 참고용. 결과와 무관하게 아래에서는 항상 Welch's를 씀
    levene_stat, levene_p = stats.levene(x1, x0)

    # (b) 왜도 — 전체 표본 기준 (이상치 영향 파악용)
    skew = pd.concat([x0, x1]).skew()

    # (c) Welch's t-test (equal_var=False 가 핵심)
    t_stat, p_val = stats.ttest_ind(x1, x0, equal_var=False)

    # (d) Cohen's d (합동표준편차 기준)
    n0, n1 = len(x0), len(x1)
    pooled_sd = np.sqrt(
        ((n0 - 1) * x0.var(ddof=1) + (n1 - 1) * x1.var(ddof=1)) / (n0 + n1 - 2)
    )
    cohens_d = (x1.mean() - x0.mean()) / pooled_sd if pooled_sd > 0 else np.nan

    return levene_p, skew, t_stat, p_val, cohens_d

## 3. A/B/C 전체 순회하며 검정 실행

In [ ]:
results = []

for group_name, path in FILES.items():
    df = pd.read_csv(path)
    continuous_cols = get_continuous_cols(df)
    print(f"[{group_name}] shape={df.shape} · 연속형 변수 {len(continuous_cols)}개: {continuous_cols}")

    for col in continuous_cols:
        x0 = df.loc[df[TARGET_COL] == 0, col].dropna()
        x1 = df.loc[df[TARGET_COL] == 1, col].dropna()

        levene_p, skew, t_stat, p_val, cohens_d = test_one_variable(x1, x0)

        results.append({
            "group": group_name,
            "variable": col,
            "n_target0": len(x0),
            "n_target1": len(x1),
            "mean_target0": x0.mean(),
            "mean_target1": x1.mean(),
            "skewness": skew,
            "skew_flag": "왜도큼(|s|>2)" if abs(skew) > SKEW_ALERT else "",
            "levene_p": levene_p,
            "equal_var_at_0.05": levene_p >= ALPHA,   # 참고용: True면 등분산 가정 기각 안 됨
            "t_stat": t_stat,
            "p_raw": p_val,
            "cohens_d": cohens_d,
        })

res_df = pd.DataFrame(results)
res_df.head()

## 4. 다중비교 보정 (FDR, Benjamini-Hochberg)

In [ ]:
reject, p_fdr, _, _ = multipletests(res_df["p_raw"], alpha=ALPHA, method="fdr_bh")
res_df["p_fdr"] = p_fdr
res_df["sig_fdr_0.05"] = reject

## 5. IV값 병합 (변수선정_최종.csv)

In [ ]:
try:
    varsel = pd.read_csv(VARSEL_PATH)
    iv_map = varsel.set_index("변수")["IV"]
    res_df["IV"] = res_df["variable"].map(iv_map)
    n_missing_iv = res_df["IV"].isna().sum()
    if n_missing_iv:
        print(f"※ IV값을 못 찾은 변수 {n_missing_iv}개 (변수선정_최종.csv에 없음) — 직접 확인 필요")
except FileNotFoundError:
    print(f"※ {VARSEL_PATH} 를 못 찾아서 IV값 없이 진행합니다. 경로를 확인해주세요.")
    res_df["IV"] = np.nan

## 6. 정리 및 저장

In [ ]:
# 효과크기(|Cohen's d|) 기준 내림차순 정렬
res_df = res_df.reindex(res_df["cohens_d"].abs().sort_values(ascending=False).index).reset_index(drop=True)

display_cols = ["group", "variable", "mean_target0", "mean_target1",
                 "t_stat", "p_raw", "p_fdr", "sig_fdr_0.05", "cohens_d",
                 "IV", "skew_flag", "levene_p"]

print("=== Welch's t-test 결과 (A/B/C 연속형 변수, |Cohen\'s d| 내림차순) ===")
display(res_df[display_cols].round(4))

n_sig = res_df["sig_fdr_0.05"].sum()
print(f"\nFDR 보정 후 유의(p_fdr < {ALPHA}): {n_sig} / {len(res_df)}개")

OUT_PATH = os.path.join(BASE_DIR, "ttest_results_ABC_final.csv")
res_df.to_csv(OUT_PATH, index=False)
print(f"결과 저장: {OUT_PATH}")

## 7. (선택) 효과크기 막대그래프

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, max(4, 0.35 * len(res_df))))
color_map = {"A_금융": "#1E2761", "B_기존비금융": "#3E5AA8", "C_신규비금융": "#F2A93B"}
colors = res_df["group"].map(color_map)
plt.barh(res_df["variable"][::-1], res_df["cohens_d"][::-1], color=colors[::-1])
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Cohen\'s d (TARGET=1 \u2212 TARGET=0)")
plt.title("A/B/C군 연속형 변수 — TARGET과의 효과크기(Welch\'s t-test 기반)")
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "ttest_cohens_d_barplot.png"), dpi=150)
plt.show()